# Notebook 2 — Dot plot: chrF++ a nivel segmento vs juicio humano

Cruza el **chrF++ por segmento** del sistema evaluado (`v2-nostrat`) con los
**juicios humanos** de una lingüista con experiencia en qom.

### Los datos de juicio humano

- Segmentos evaluados en **ambas direcciones** (`qom2es` y `es2qom`), en un mismo CSV
  (una fila por segmento y dirección; la dirección va en la columna `direction`).
- Las oraciones **se eligieron sin garantizar que vinieran de un split de test**
  controlado, así que su `segment_id` se recupera por texto contra el corpus `qomL-hf`
  completo, y la hipótesis se toma de la columna con la predicción del sistema.
- Juicios **categóricos** (aproximadamente: correcto / incorrecto / dudoso).
- **No siguen el esquema MQM**: la tipificación MQM es una relectura posterior, así que
  **no hay severidades ni puntajes numéricos**.

> **No** se inventa un escalar de calidad ni se deriva uno a partir de las categorías.
> El origen no controlado de los segmentos es una limitación a declarar en la lectura.

### La pregunta de la figura

¿Los segmentos juzgados **incorrectos** se separan de los **correctos** en el eje de
chrF++, o se mezclan?

## Celda de configuración

Requiere que la **Notebook 1** haya corrido antes (produce `results/chrf_por_segmento.csv`
y `results/traducciones_tidy.csv`). **Completá `JUICIOS_CSV`** (columnas `segment_id,
direction, juicio_humano`) y, si querés, fijá `SISTEMA_EVALUADO`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 20260729
np.random.seed(RANDOM_STATE)

# ── Entradas ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
# CSV de juicios. Puede NO traer segment_id: se recupera por texto desde qomL-hf.
# Columnas mínimas: direction, juicio (+ texto qom/es para recuperar el id).
JUICIOS_CSV = DATA_DIR / "human_eval/v2-nostrat.csv"

# Corpus completo empaquetado (parquet por config y split). De acá se recuperan
# los segment_id de TODOS los segmentos (no sólo del test común de Base).
QOML_HF_DIR = DATA_DIR / "qomL-hf"

# Salidas de la Notebook 1 (opcionales acá: sólo se usan como atajo si existen).
RESULTS_DIR = Path("data/results")
SEGMENT_CHRF_CSV = RESULTS_DIR / "chrf_por_segmento.csv"
TIDY_CSV         = RESULTS_DIR / "traducciones_tidy.csv"

FIG_DIR = Path("poster/figures")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# n mínimo por categoría para calcular correlaciones/tests.
N_MINIMO_TESTS = 10
N_BOOTSTRAP = 100

# Sistema cuyas salidas evaluó la lingüista.
SISTEMA_EVALUADO = "qom-mt-v2-aleatorio"

# ── Hipótesis del sistema evaluado ────────────────────────────────────────────
# Las oraciones evaluadas pueden NO estar en el test común, así que no siempre hay
# chrF++ precalculado. La hipótesis se obtiene, en este orden de preferencia:
#
#   1) COLUMNA INLINE del CSV de juicios: si el CSV ya trae la predicción del
#      sistema (p. ej. la columna "v2-nostrat"), se usa esa. Es lo más fiel:
#      es exactamente el texto que vio la lingüista. None = autodetectar.
COL_PREDICCION_INLINE = None
#
#   2) CORRER EL MODELO: si no hay columna inline, se generan las hipótesis con el
#      checkpoint de v2-nostrat sobre el texto fuente. Completá el/los checkpoint(s).
CHECKPOINT_EVALUADO = None    # str (un modelo p/ambas direcciones) o
                              # {"qom2es": "ruta/ckpt", "es2qom": "ruta/ckpt"}
GEN_PARAMS = dict(num_beams=4, no_repeat_ngram_size=3, max_new_tokens=128)
LANG = {"qom": "grn_Latn", "es": "spa_Latn"}
DIRECTION_LANGS = {"qom2es": ("qom", "es"), "es2qom": ("es", "qom")}

print("Config Notebook 2 cargada. Sistema evaluado:", SISTEMA_EVALUADO)

## 2.1 — Datos: recuperación de `segment_id`, hipótesis y chrF++

El CSV de juicios **no trae `segment_id`** y las oraciones evaluadas **pueden no estar en el
test común** (se eligieron sin garantizar que vinieran de un split de test). Por eso:

1. **Recuperamos `segment_id` contra `qomL-hf` completo** (todos los `config`/`split`), no
   sólo el test de Base. El match es sobre texto **normalizado** (NFC, minúsculas, sin
   puntuación) en cuatro niveles: par `(qom, es)` → sólo qom → sólo es → difuso (para
   typos, listado aparte). El texto gold pasa a tomarse de `qomL-hf`.
2. **Obtenemos la hipótesis de `v2-nostrat`**: si el CSV ya trae la columna con la
   predicción (lo más fiel), se usa; si no, se **corre el modelo** sobre el texto fuente.
3. **Calculamos chrF++ por segmento** desde `(hipótesis, referencia)` y cruzamos con el
   juicio humano.

> No es lo más riguroso (las oraciones no salieron de un test controlado), pero es lo
> disponible. Todo lo que no matchea o se genera queda **explícitamente reportado**.

In [ ]:
import re
import unicodedata

# ── Normalización de direcciones (misma convención que la Notebook 1) ─────────
DIRECTION_ALIASES = {
    "qom2es": "qom2es", "qom-es": "qom2es", "qom_es": "qom2es", "qomes": "qom2es",
    "qom->es": "qom2es", "qom→es": "qom2es", "tob2spa": "qom2es",
    "es2qom": "es2qom", "es-qom": "es2qom", "es_qom": "es2qom", "esqom": "es2qom",
    "es->qom": "es2qom", "es→qom": "es2qom", "spa2tob": "es2qom",
}
def norm_direction(v):
    k = str(v).strip().lower().replace(" ", "")
    if k in DIRECTION_ALIASES:
        return DIRECTION_ALIASES[k]
    raise ValueError(f"Dirección no reconocida: {v!r}. Agregala a DIRECTION_ALIASES.")

# ── Normalización de texto (para el matching de segment_id) ───────────────────
_PUNT = re.compile(r"[^\w\s]", flags=re.UNICODE)
_ESP  = re.compile(r"\s+")
def normalizar_texto(t):
    t = unicodedata.normalize("NFC", str(t)).lower()
    t = _PUNT.sub(" ", t)
    t = _ESP.sub(" ", t).strip()
    return t

# ── Categorías de juicio (normalizadas) ───────────────────────────────────────
CATEGORIAS_ESPERADAS = {"correcto", "incorrecto", "dudoso"}
CAT_ALIASES = {
    "correcto": "correcto", "correcta": "correcto", "ok": "correcto", "bien": "correcto",
    "si": "correcto", "sí": "correcto", "1": "correcto",
    "incorrecto": "incorrecto", "incorrecta": "incorrecto", "mal": "incorrecto",
    "no": "incorrecto", "0": "incorrecto",
    "dudoso": "dudoso", "dudosa": "dudoso", "duda": "dudoso", "?": "dudoso",
    "parcial": "dudoso",
}
def norm_categoria(v):
    return CAT_ALIASES.get(str(v).strip().lower(), str(v).strip().lower())

# ── Nombres de columna candidatos para la predicción inline del sistema ───────
def _candidatos_col_pred(sistema):
    s = sistema.lower().replace("qom-mt-", "").replace("qom_mt_", "")
    s = s.replace("aleatorio", "nostrat").replace("estratificado", "strat")
    cands = [sistema, s, s.replace("-", "_"), s.replace("_", "-")]
    vistos, out = set(), []
    for c in cands:
        if c and c not in vistos:
            vistos.add(c); out.append(c)
    return out

# ── Carga del CSV de juicios ──────────────────────────────────────────────────
jr = pd.read_csv(JUICIOS_CSV)
lower = {c.lower().strip(): c for c in jr.columns}
def _col(*cands, req=True):
    for c in cands:
        if c in lower:
            return lower[c]
    if req:
        raise ValueError(f"Falta alguna de estas columnas en {JUICIOS_CSV}: {cands}")
    return None

col_dir    = _col("direction", "dir", "sentido", "direccion", "dirección")
col_juicio = _col("juicio_humano", "juicio", "categoria", "categoría", "label",
                  "evaluacion", "evaluación")
col_qom    = _col("qom", "toba", "tob", "qom_text", req=False)
col_es     = _col("es", "spa", "español", "espanol", "castellano", req=False)
col_seg    = _col("segment_id", "id", "seg_id", req=False)

j = pd.DataFrame({
    "direction":     jr[col_dir].map(norm_direction),
    "juicio_humano": jr[col_juicio].map(norm_categoria),
})
if col_qom is not None: j["qom"] = jr[col_qom].astype("string")
if col_es  is not None: j["es"]  = jr[col_es].astype("string")
if col_seg is not None: j["segment_id"] = jr[col_seg]

# Predicción inline del sistema evaluado (si el CSV la trae).
if COL_PREDICCION_INLINE is not None:
    key = COL_PREDICCION_INLINE.lower().strip()
    if key not in lower:
        raise ValueError(f"No encuentro la columna inline '{COL_PREDICCION_INLINE}'. "
                         f"Columnas: {list(jr.columns)}")
    col_pred = lower[key]
else:
    col_pred = None
    for c in _candidatos_col_pred(SISTEMA_EVALUADO):
        if c in lower:
            col_pred = lower[c]; break
if col_pred is not None:
    j["hyp_inline"] = jr[col_pred].astype("string")

inesperadas = set(j["juicio_humano"]) - CATEGORIAS_ESPERADAS
if inesperadas:
    print(f"[aviso] Categorías fuera de {CATEGORIAS_ESPERADAS}: {inesperadas}.")
print("Filas de juicio:", len(j))
print(j.groupby(["direction", "juicio_humano"]).size().to_string())
print("\nColumna de predicción inline:", col_pred if col_pred else "(no hay → se correrá el modelo)")
print("Texto disponible para recuperar id:", [c for c in ("qom", "es") if c in j.columns])

In [ ]:
# ── Recuperar segment_id desde qomL-hf (todo el corpus) ───────────────────────
from difflib import SequenceMatcher

UMBRAL_DIFUSO = 0.90   # similitud mínima para aceptar un match difuso (revisar a mano)

def _config_preferida(sistema):
    s = sistema.lower().replace("qom-mt-", "").replace("qom_mt_", "")
    return s.replace("aleatorio", "nostrat").replace("estratificado", "strat")

def _indice_qomL_hf():
    parquets = sorted(QOML_HF_DIR.glob("*/*.parquet"))
    if not parquets:
        raise FileNotFoundError(
            f"No hay parquets en {QOML_HF_DIR}/*/*.parquet. Descargá el corpus qomL-hf.")
    trozos = []
    for p in parquets:
        df = pd.read_parquet(p)
        if not {"qom", "es"} <= set(df.columns):
            print(f"[aviso] {p} sin columnas qom/es: se saltea."); continue
        sub = df[["qom", "es"] + (["id"] if "id" in df.columns else [])].copy()
        sub["hf_id"]  = sub["id"] if "id" in sub.columns else pd.NA
        sub["config"] = p.parent.name
        sub["split"]  = p.stem
        trozos.append(sub[["hf_id", "config", "split", "qom", "es"]])
    idx = pd.concat(trozos, ignore_index=True).dropna(subset=["qom", "es"])
    idx["qom_norm"] = idx["qom"].map(normalizar_texto)
    idx["es_norm"]  = idx["es"].map(normalizar_texto)
    # Preferimos la config del sistema evaluado al deduplicar pares repetidos.
    pref = _config_preferida(SISTEMA_EVALUADO)
    idx["_pref"] = (~idx["config"].str.contains(pref, case=False, na=False)).astype(int)
    idx = (idx.sort_values(["_pref", "config", "split"])
              .drop_duplicates(["qom_norm", "es_norm"], keep="first")
              .reset_index(drop=True))
    idx["segment_id"] = [
        r.hf_id if pd.notna(r.hf_id) else f"{r.config}:{r.split}:{i}"
        for i, r in enumerate(idx.itertuples(index=False))]
    return idx

def recuperar_segment_ids(j):
    idx = _indice_qomL_hf()
    by_pair, by_qom, by_es, reg = {}, {}, {}, {}
    for r in idx.itertuples(index=False):
        by_pair.setdefault((r.qom_norm, r.es_norm), r.segment_id)
        by_qom.setdefault(r.qom_norm, set()).add(r.segment_id)
        by_es.setdefault(r.es_norm, set()).add(r.segment_id)
        reg[r.segment_id] = (r.qom, r.es, r.config, r.split)
    universo_qom = list(by_qom)

    tiene_qom, tiene_es = "qom" in j.columns, "es" in j.columns
    if not (tiene_qom or tiene_es):
        raise ValueError("El CSV de juicios no trae segment_id ni texto (qom/es).")

    out = j.copy()
    sids, met, sc, cfg, spl, gq, ge = [], [], [], [], [], [], []
    for r in j.itertuples(index=False):
        qn = normalizar_texto(r.qom) if tiene_qom and pd.notna(r.qom) else None
        en = normalizar_texto(r.es)  if tiene_es  and pd.notna(r.es)  else None
        sid, via, s = None, "sin_match", float("nan")
        if qn and en and (qn, en) in by_pair:
            sid, via, s = by_pair[(qn, en)], "par_exacto", 1.0
        elif qn and len(by_qom.get(qn, ())) == 1:
            sid, via, s = next(iter(by_qom[qn])), "solo_qom", 1.0
        elif en and len(by_es.get(en, ())) == 1:
            sid, via, s = next(iter(by_es[en])), "solo_es", 1.0
        elif qn:
            mejor, mejor_s = None, 0.0
            for cand in universo_qom:
                v = SequenceMatcher(None, qn, cand).ratio()
                if v > mejor_s:
                    mejor, mejor_s = cand, v
            if mejor is not None and mejor_s >= UMBRAL_DIFUSO and len(by_qom[mejor]) == 1:
                sid, via, s = next(iter(by_qom[mejor])), "difuso_qom", round(mejor_s, 3)
        sids.append(sid); met.append(via); sc.append(s)
        q, e, c, sp = reg.get(sid, (None, None, None, None))
        gq.append(q); ge.append(e); cfg.append(c); spl.append(sp)
    out["segment_id"] = sids
    out["match_metodo"] = met
    out["match_score"] = sc
    out["hf_config"] = cfg
    out["hf_split"] = spl
    # El texto gold pasa a ser el de qomL-hf (referencia limpia para chrF++).
    out["qom"] = [g if g is not None else o for g, o in zip(gq, out.get("qom", [None]*len(out)))]
    out["es"]  = [g if g is not None else o for g, o in zip(ge, out.get("es",  [None]*len(out)))]
    return out

if "segment_id" in j.columns and j["segment_id"].notna().all():
    print("El CSV ya trae segment_id: se usa tal cual (no se recupera por texto).")
    j["match_metodo"] = "provisto"; j["match_score"] = 1.0
    j["hf_config"] = pd.NA; j["hf_split"] = pd.NA
else:
    j = recuperar_segment_ids(j)

# ── Informe del matching ──────────────────────────────────────────────────────
print("Método de recuperación de segment_id:")
print(j["match_metodo"].value_counts().to_string())

difusos = j[j["match_metodo"] == "difuso_qom"]
if len(difusos):
    print(f"\n[revisar] {len(difusos)} match(es) DIFUSO(s) — verificá el texto a mano:")
    print(difusos[[c for c in ["direction", "segment_id", "match_score", "qom", "es"]
                   if c in difusos.columns]].to_string(index=False))

sin = j[j["segment_id"].isna()]
if len(sin):
    print(f"\n[aviso] {len(sin)} fila(s) SIN segment_id. Bajá UMBRAL_DIFUSO o revisá el texto:")
    print(sin[[c for c in ["direction", "juicio_humano", "qom", "es"]
               if c in sin.columns]].to_string(index=False))

JUICIOS_CON_ID = RESULTS_DIR / "juicios_con_id.csv"
j.to_csv(JUICIOS_CON_ID, index=False)
print(f"\nJuicios con segment_id -> {JUICIOS_CON_ID}")

j = j.dropna(subset=["segment_id"]).reset_index(drop=True)

In [ ]:
from sacrebleu.metrics import CHRF
chrf_pp = CHRF(word_order=2)   # chrF++ (char_order=6, beta=2)

# ── Texto FUENTE y REFERENCIA por fila, según la dirección ────────────────────
es_q2e = j["direction"].eq("qom2es")
j["source"]    = np.where(es_q2e, j["qom"], j["es"])
j["reference"] = np.where(es_q2e, j["es"], j["qom"])

# ── Hipótesis del sistema evaluado ────────────────────────────────────────────
# (1) columna inline (lo más fiel: es lo que vio la lingüista).
if "hyp_inline" in j.columns and j["hyp_inline"].notna().any():
    j["hypothesis"] = j["hyp_inline"].astype("string")
    print(f"Hipótesis desde columna inline: {j['hypothesis'].notna().sum()}/{len(j)} filas.")
else:
    j["hypothesis"] = pd.Series([pd.NA] * len(j), dtype="string")
    print("No hay columna inline de predicción.")

# (2) atajo: preds ya calculadas por la NB1 (sólo cubren el test común).
if j["hypothesis"].isna().any() and Path(TIDY_CSV).exists():
    tidy = pd.read_csv(TIDY_CSV)
    tidy["direction"] = tidy["direction"].map(norm_direction)
    t = (tidy[tidy["system"] == SISTEMA_EVALUADO][["segment_id", "direction", "hypothesis"]]
         .rename(columns={"hypothesis": "_hyp_tidy"}))
    j = j.merge(t, on=["segment_id", "direction"], how="left")
    falta = j["hypothesis"].isna()
    j.loc[falta, "hypothesis"] = j.loc[falta, "_hyp_tidy"].astype("string")
    j = j.drop(columns=["_hyp_tidy"])

print(f"Hipótesis faltantes (requieren correr el modelo): {int(j['hypothesis'].isna().sum())}")

In [ ]:
# ── Correr el modelo v2-nostrat para las hipótesis faltantes ──────────────────
# Genera SOLO las que faltan, sobre el texto FUENTE de cada fila, con las mismas
# etiquetas de idioma y GEN_PARAMS que la Notebook 1. Requiere `transformers` +
# `torch` y el checkpoint en CHECKPOINT_EVALUADO. Idealmente, generación
# determinística (beam search) para reproducir lo que vio la lingüista.
falta = j["hypothesis"].isna()
if not falta.any():
    print("No falta generar hipótesis: se saltea el modelo.")
elif CHECKPOINT_EVALUADO is None:
    print(f"[aviso] Faltan {int(falta.sum())} hipótesis y CHECKPOINT_EVALUADO es None.\n"
          "  → Completá el checkpoint de v2-nostrat en la config, o agregá al CSV de\n"
          "    juicios la columna con la predicción del sistema (COL_PREDICCION_INLINE).")
else:
    import torch
    from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Generando {int(falta.sum())} hipótesis en {DEVICE}. GEN_PARAMS={GEN_PARAMS}")

    def _ckpt(direccion):
        return CHECKPOINT_EVALUADO[direccion] if isinstance(CHECKPOINT_EVALUADO, dict) \
               else CHECKPOINT_EVALUADO

    def _load(name):
        tok = NllbTokenizer.from_pretrained(name)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(name).to(DEVICE).eval()
        return mdl, tok

    def _translate(mdl, tok, textos, src_lang, tgt_lang, batch=8):
        tok.src_lang = src_lang
        forced = tok.convert_tokens_to_ids(tgt_lang)
        res = []
        for i in range(0, len(textos), batch):
            b = [str(x) for x in textos[i:i + batch]]
            enc = tok(b, return_tensors="pt", padding=True, truncation=True,
                      max_length=GEN_PARAMS["max_new_tokens"]).to(DEVICE)
            with torch.no_grad():
                gen = mdl.generate(**enc, forced_bos_token_id=forced, **GEN_PARAMS)
            res += tok.batch_decode(gen, skip_special_tokens=True)
        return res

    cache = {}
    for direccion in ["qom2es", "es2qom"]:
        m = falta & j["direction"].eq(direccion)
        if not m.any():
            continue
        src_l, tgt_l = DIRECTION_LANGS[direccion]
        ck = _ckpt(direccion)
        if ck not in cache:
            cache[ck] = _load(ck)
        mdl, tok = cache[ck]
        hyps = _translate(mdl, tok, j.loc[m, "source"].tolist(), LANG[src_l], LANG[tgt_l])
        j.loc[m, "hypothesis"] = pd.array(hyps, dtype="string")
    for mdl, tok in cache.values():
        del mdl, tok
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"Generación lista. Hipótesis faltantes ahora: {int(j['hypothesis'].isna().sum())}")

In [ ]:
# ── chrF++ por segmento (hyp vs ref) y armado de `datos` ──────────────────────
listo = j.dropna(subset=["hypothesis", "reference"]).copy()
listo = listo[listo["hypothesis"].astype(str).str.len() > 0]
listo["chrf_segment"] = [
    round(chrf_pp.sentence_score(str(h), [str(r)]).score, 4)
    for h, r in zip(listo["hypothesis"], listo["reference"])
]

datos = (listo[["segment_id", "direction", "juicio_humano", "chrf_segment",
                "source", "reference", "hypothesis"]]
         .reset_index(drop=True))
sistema = SISTEMA_EVALUADO   # lo usan las celdas siguientes

n_fuera = len(j) - len(datos)
if n_fuera:
    print(f"[aviso] {n_fuera} fila(s) sin hipótesis quedaron fuera del cruce "
          "(faltó generarlas o venían vacías).")
print(f"Segmentos con chrF++ para el dot plot: {len(datos)} (sistema: {sistema}).")
if len(datos):
    print(datos.groupby(["direction", "juicio_humano"]).size().to_string())
datos.to_csv(RESULTS_DIR / "juicio_vs_chrf.csv", index=False)

## 2.2 — Figura principal (dot plot)

Un panel por dirección:

- eje **Y**: chrF++ del segmento;
- eje **X**: categoría de juicio humano, con **jitter** horizontal;
- **color y forma** de marcador según categoría (paleta apta para daltonismo);
- **mediana** de cada grupo con línea horizontal;
- **puntos individuales visibles**: no boxplot ni violín (con este n no corresponde).

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 18, "axes.titlesize": 22, "axes.labelsize": 20,
    "xtick.labelsize": 16, "ytick.labelsize": 16, "legend.fontsize": 16,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.constrained_layout.use": True,
})

# Okabe-Ito: color + forma redundantes por categoría (accesibilidad).
ESTILO_CAT = {
    "correcto":   {"color": "#009E73", "marker": "o"},   # verde, círculo
    "dudoso":     {"color": "#E69F00", "marker": "^"},   # naranja, triángulo
    "incorrecto": {"color": "#D55E00", "marker": "s"},   # bermellón, cuadrado
}
ORDEN_CAT = ["incorrecto", "dudoso", "correcto"]

def guardar(fig, nombre):
    fig.savefig(FIG_DIR / f"{nombre}.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{nombre}.png", bbox_inches="tight", dpi=300)
    print(f"  guardada: {nombre}.pdf y .png (300 dpi)")

rng = np.random.default_rng(RANDOM_STATE)
direcciones = [d for d in ["qom2es", "es2qom"] if d in set(datos["direction"])]

fig, axes = plt.subplots(1, len(direcciones), figsize=(7 * len(direcciones), 7),
                         sharey=True, squeeze=False)
axes = axes[0]
for ax, direccion in zip(axes, direcciones):
    d = datos[datos["direction"] == direccion]
    cats = [c for c in ORDEN_CAT if c in set(d["juicio_humano"])]
    cats += [c for c in sorted(set(d["juicio_humano"])) if c not in cats]
    for x, cat in enumerate(cats):
        g = d[d["juicio_humano"] == cat]
        est = ESTILO_CAT.get(cat, {"color": "#0072B2", "marker": "D"})
        jitter = rng.uniform(-0.18, 0.18, size=len(g))
        ax.scatter(np.full(len(g), x) + jitter, g["chrf_segment"],
                   color=est["color"], marker=est["marker"], s=130,
                   edgecolor="white", linewidth=0.7, alpha=0.9, zorder=3)
        med = g["chrf_segment"].median()
        ax.plot([x - 0.28, x + 0.28], [med, med], color="#222222", lw=3, zorder=4)
        ax.text(x, -6, f"n={len(g)}", ha="center", va="top", fontsize=13, color="#555")
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats)
    ax.set_title(direccion); ax.set_xlabel("juicio humano")
    ax.set_ylim(-8, 105)
axes[0].set_ylabel("chrF++ (segmento)")
fig.suptitle(f"chrF++ por segmento vs juicio humano — {sistema}")
guardar(fig, "figura_dotplot_juicio_chrf")
plt.show()

## 2.3 — Estadística descriptiva, con cautela

- **n por categoría y dirección.** Si alguna categoría tiene **menos de 10 ítems**, no se
  calculan coeficientes de correlación ni tests de hipótesis: se imprime una advertencia
  explícita.
- Sí se reportan **mediana, rango y rango intercuartílico** de chrF++ por categoría.
- Toda medida de asociación que se calcule va con su **IC bootstrap**.

In [ ]:
# ── Descriptivos por dirección x categoría ────────────────────────────────────
filas = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    for cat, g in d.groupby("juicio_humano"):
        x = g["chrf_segment"]
        filas.append({"direction": direccion, "categoria": cat, "n": len(g),
                      "mediana": round(x.median(), 2), "min": round(x.min(), 2),
                      "max": round(x.max(), 2), "q1": round(x.quantile(0.25), 2),
                      "q3": round(x.quantile(0.75), 2),
                      "iqr": round(x.quantile(0.75) - x.quantile(0.25), 2)})
descriptivos = pd.DataFrame(filas).sort_values(["direction", "categoria"])
print(descriptivos.to_string(index=False))
descriptivos.to_csv(RESULTS_DIR / "descriptivos_por_categoria.csv", index=False)

In [ ]:
# ── Asociación juicio-chrF++, sólo si el n lo permite ─────────────────────────
# Codificamos el juicio ordinalmente SÓLO para medir asociación monotónica (Spearman);
# esto no implica inventar un escalar de calidad, es un rango de las 3 categorías.
# Spearman = Pearson sobre rangos; lo implementamos con numpy para no sumar scipy.
ORDEN_ORDINAL = {"incorrecto": 0, "dudoso": 1, "correcto": 2}

def _rankdata(a):
    a = np.asarray(a, dtype=float)
    orden = a.argsort(kind="mergesort")
    r = np.empty(len(a), dtype=float)
    sa = a[orden]
    i = 0
    while i < len(a):
        j = i
        while j + 1 < len(a) and sa[j + 1] == sa[i]:
            j += 1
        r[orden[i:j + 1]] = (i + j) / 2.0 + 1.0
        i = j + 1
    return r

def spearman(x, y):
    rx, ry = _rankdata(x), _rankdata(y)
    if rx.std() == 0 or ry.std() == 0:
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])

def spearman_ic(x, y, n_boot, rng):
    rho = spearman(x, y)
    idx = np.arange(len(x))
    reps = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        r = spearman(x[s], y[s])
        if not np.isnan(r):
            reps.append(r)
    lo, hi = np.percentile(reps, [2.5, 97.5]) if reps else (np.nan, np.nan)
    return rho, lo, hi

rng_b = np.random.default_rng(RANDOM_STATE + 7)
filas_assoc = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    conteos = d["juicio_humano"].value_counts()
    categorias_ok = all(conteos.get(c, 0) >= N_MINIMO_TESTS for c in ORDEN_ORDINAL)
    usables = d[d["juicio_humano"].isin(ORDEN_ORDINAL)]
    if not categorias_ok:
        chicas = {c: int(conteos.get(c, 0)) for c in ORDEN_ORDINAL}
        print(f"[{direccion}] n por categoría {chicas}: alguna < {N_MINIMO_TESTS}. "
              "NO se calculan correlaciones ni tests de hipótesis "
              "— el tamaño de muestra no lo permite.")
        continue
    x = usables["juicio_humano"].map(ORDEN_ORDINAL).to_numpy()
    y = usables["chrf_segment"].to_numpy()
    rho, lo, hi = spearman_ic(x, y, N_BOOTSTRAP, rng_b)
    filas_assoc.append({"direction": direccion, "spearman_rho": round(rho, 3),
                        "ic_low": round(lo, 3), "ic_high": round(hi, 3),
                        "n": len(usables)})

if filas_assoc:
    assoc = pd.DataFrame(filas_assoc)
    print(assoc.to_string(index=False))
    assoc.to_csv(RESULTS_DIR / "asociacion_spearman.csv", index=False)
else:
    print("No se calcularon medidas de asociación (ver avisos arriba).")

## 2.4 — Casos para inspección cualitativa

Segmentos **juzgados correctos con chrF++ más bajo** y **juzgados incorrectos con chrF++
más alto** (5 de cada uno por dirección), con fuente, referencia e hipótesis. Sirven como
ejemplos para el póster (dónde la métrica y el juicio humano se contradicen).

In [ ]:
# ── Casos donde métrica y juicio se contradicen (desde `datos`) ───────────────
casos_out = []
for direccion in direcciones:
    d = datos[datos["direction"] == direccion]
    correctos   = d[d["juicio_humano"] == "correcto"].nsmallest(5, "chrf_segment")
    incorrectos = d[d["juicio_humano"] == "incorrecto"].nlargest(5, "chrf_segment")
    for etiqueta, sub in [("correcto_chrf_bajo", correctos),
                          ("incorrecto_chrf_alto", incorrectos)]:
        s = sub.copy(); s.insert(0, "caso", etiqueta); casos_out.append(s)

if casos_out:
    casos = pd.concat(casos_out, ignore_index=True)
    cols = [c for c in ["caso", "direction", "segment_id", "juicio_humano", "chrf_segment",
                        "source", "reference", "hypothesis"] if c in casos.columns]
    casos = casos[cols]
    print(casos.to_string(index=False))
    casos.to_csv(RESULTS_DIR / "casos_cualitativos.csv", index=False)
    print(f"\nGuardado -> {RESULTS_DIR / 'casos_cualitativos.csv'}")
else:
    print("Sin casos para inspección (datos vacío).")

### Cierre

- La figura contesta de un vistazo si **incorrectos** y **correctos** se separan o se
  mezclan en el eje chrF++.
- Con estos tamaños de muestra, la estadística es **descriptiva**: correlaciones y tests
  sólo si cada categoría llega a `N_MINIMO_TESTS`, siempre con IC bootstrap.
- No se derivó ningún escalar de calidad a partir de los juicios categóricos.